# K-Means Clustering Lab — Chinatown HEROS Project

**Chinatown HEROS (Health & Environmental Research in Open Spaces)**  
**Lab Duration:** 50 minutes | **Dataset:** 48,123 observations across 12 open-space sites  
**Study Period:** July 19 – August 23, 2023 | **Resolution:** 10-minute intervals

---

## What You'll Learn

In this lab, we'll use **k-means clustering** to discover natural groupings among Chinatown's 12 open-space monitoring sites based on their environmental profiles. By the end, you'll be able to:

1. **Explore** a real-world environmental dataset and identify clustering opportunities
2. **Visualize** multi-dimensional data with insight-driven charts (scatter plots, heatmaps, radar charts)
3. **Implement** k-means step-by-step: standardize → elbow method → silhouette analysis → fit → interpret
4. **Interpret** what each cluster means in environmental and public health terms
5. **Export** results for use in PowerBI dashboards

### Why Clustering?

We've already conducted extensive analysis on this dataset (PM2.5 comparisons, temperature validation, AQI analysis, temporal patterns). But one question remains: **can we group the 12 sites into meaningful categories based on their overall environmental profile?** This is exactly what k-means clustering answers.

---

# Step 1 — Data Understanding & Clustering Opportunities

Before applying any algorithm, we need to deeply understand what we're working with. This section loads the HEROS dataset, examines the key variables, and identifies which features are good candidates for clustering.

> **💡 Teaching Moment:** In real-world data science, you spend ~80% of your time understanding and preparing data. The algorithm is the easy part — choosing the _right features_ and _interpreting results_ is what separates good analysis from great analysis.

### 1.1 — Import Libraries & Load Data

We'll use **pandas** for data manipulation, **plotly** for interactive visualizations, and **scikit-learn** for the clustering algorithm.

In [18]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# Load the cleaned HEROS dataset
df = pd.read_csv('../data/clean/data_HEROS_clean.csv')
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Sites: {df['site_id'].nunique()}")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")
df.head(3)

Dataset shape: 48,123 rows × 46 columns
Sites: 12
Date range: 2023-07-19 16:40:00 to 2023-08-23 15:50:00


,site_id,datetime,kes_mean_temp_f,kes_mean_wbgt_f,kes_mean_humid_pct,kes_mean_press_inHg,kes_mean_heat_f,kes_mean_dew_f,pa_mean_pm2_5_atm_b_corr_2,mean_temp_out_f,...,Roads_Area_Percent_25m,Greenspace_Area_Percent_25m,Trees_Area_Percent_25m,Impervious_Area_Percent_25m,Industrial_Area_Percent_25m,Roads_Area_Percent_50m,Greenspace_Area_Percent_50m,Trees_Area_Percent_50m,Impervious_Area_Percent_50m,Industrial_Area_Percent_50m
0,berkley,2023-07-19 19:00:00,84.6,67.1,40.6,29.95,84.4,58.0,15.140488,86.05,...,0.466609,0.259756,0.274022,0.658449,0.0,0.278347,0.168168,0.338126,0.625741,0.0
1,berkley,2023-07-19 19:10:00,85.3,67.5,40.0,29.96,85.3,58.3,14.550256,85.60,...,0.466609,0.259756,0.274022,0.658449,0.0,0.278347,0.168168,0.338126,0.625741,0.0
2,berkley,2023-07-19 19:20:00,82.9,66.9,43.7,29.95,82.2,58.6,14.998800,85.35,...,0.466609,0.259756,0.274022,0.658449,0.0,0.278347,0.168168,0.338126,0.625741,0.0


### 1.2 — What Do We Already Know?

From our previous research phases, we've established several key findings:

| Finding | Source |
|---------|--------|
| Purple Air PM2.5 reads +0.7 to +2.7 µg/m³ higher than DEP FEM | Q1 Analysis |
| ~3 µg/m³ spread in mean PM2.5 across the 12 sites (7.9 to 10.8) | Phase 2 EDA |
| Temperature varies by only ~1.4°F across sites — much less than PM2.5 | Q2 Analysis |
| PM2.5 peaks at noon; WBGT peaks at 5 PM — a critical 5-hour offset | Q8 Temporal |
| Castle Square is warmest (75.3°F) but has low PM2.5 (8.2 µg/m³) | Phase 2 EDA |
| One Greenway has highest PM2.5 (10.7 µg/m³) despite having the most greenspace | Phase 2 EDA |

**The key question:** Can we formalize these site differences into distinct groups using clustering?

> **💡 Teaching Moment:** Clustering is not just a statistical exercise — it's a way to compress complex, multi-dimensional data into actionable categories. Health officials need to know: "Which sites behave similarly?" rather than memorizing 12 individual profiles.

### 1.3 — Build Site-Level Profiles

For clustering, we need **one row per site** with multiple features. We'll aggregate the 48,123 raw observations into 12 site-level means. These are our clustering features:

- **PM2.5** (µg/m³) — air quality exposure
- **Temperature** (°F) — thermal comfort
- **WBGT** (°F) — wet-bulb globe temperature (heat stress indicator)
- **Humidity** (%) — moisture level

Why these four? They capture the two main environmental justice concerns: **air pollution** and **heat exposure** — the core of the HEROS project.

In [19]:
# Aggregate raw observations to site-level means
site_profiles = df.groupby('site_id').agg(
    pm25_mean   = ('pa_mean_pm2_5_atm_b_corr_2', 'mean'),
    temp_mean   = ('kes_mean_temp_f', 'mean'),
    wbgt_mean   = ('kes_mean_wbgt_f', 'mean'),
    humid_mean  = ('kes_mean_humid_pct', 'mean'),
    pm25_std    = ('pa_mean_pm2_5_atm_b_corr_2', 'std'),
    n_obs       = ('pa_mean_pm2_5_atm_b_corr_2', 'count'),
).round(2)

# Human-readable site names
site_names = {
    'berkley': 'Berkeley Garden', 'castle': 'Castle Square', 'chin': 'Chin Park',
    'dewey': 'Dewey Square', 'eliotnorton': 'Eliot Norton', 'greenway': 'One Greenway',
    'lyndenboro': 'Lyndboro Park', 'msh': 'Mary Soo Hoo', 'oxford': 'Oxford Plaza',
    'reggie': 'Reggie Wong', 'taitung': 'Tai Tung Park', 'tufts': 'Tufts Garden'
}
site_profiles['site_name'] = site_profiles.index.map(site_names)

print(f"Site profiles: {site_profiles.shape[0]} sites × {site_profiles.shape[1]} features\n")
site_profiles[['site_name', 'pm25_mean', 'temp_mean', 'wbgt_mean', 'humid_mean', 'pm25_std', 'n_obs']]

Site profiles: 12 sites × 7 features



,site_name,pm25_mean,temp_mean,wbgt_mean,humid_mean,pm25_std,n_obs
site_id,,,,,,,
berkley,Berkeley Garden,9.53,74.41,66.10,67.18,4.94,2445
castle,Castle Square,8.17,75.31,66.76,66.81,5.42,3793
chin,Chin Park,10.49,75.02,66.01,64.28,5.93,2199
dewey,Dewey Square,9.69,74.57,65.92,65.74,5.50,4889
eliotnorton,Eliot Norton,9.29,73.93,65.48,66.09,4.89,3888
greenway,One Greenway,10.71,74.50,65.73,65.65,6.04,4893
lyndenboro,Lyndboro Park,10.68,74.37,65.83,66.29,5.69,4786
msh,Mary Soo Hoo,9.07,73.91,65.09,64.67,4.54,4177
oxford,Oxford Plaza,7.93,74.96,65.60,63.09,3.61,2879


### 1.4 — Feature Correlations

Before clustering, let's understand how our four features relate to each other. If two features are highly correlated, they carry redundant information.

> **💡 Teaching Moment:** K-means treats each feature equally. If you include two highly correlated features, you're effectively double-weighting that dimension. Check correlations first!

In [20]:
# Correlation heatmap of our 4 clustering features
features = ['pm25_mean', 'temp_mean', 'wbgt_mean', 'humid_mean']
corr = site_profiles[features].corr().round(3)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    labels=dict(color='Pearson r'),
    x=['PM2.5', 'Temperature', 'WBGT', 'Humidity'],
    y=['PM2.5', 'Temperature', 'WBGT', 'Humidity'],
    title='Feature Correlation Matrix (Site-Level Means)'
)
fig.update_layout(width=500, height=450, template='plotly_white')
fig.show()

print("\nKey observations:")
print(f"  • PM2.5 ↔ Temperature: r = {corr.loc['pm25_mean','temp_mean']:.3f} (negative — polluted sites tend to be cooler)")
print(f"  • WBGT ↔ Humidity: r = {corr.loc['wbgt_mean','humid_mean']:.3f} (positive — WBGT incorporates humidity)")
print(f"  • PM2.5 ↔ WBGT: r = {corr.loc['pm25_mean','wbgt_mean']:.3f} (near zero — independent dimensions!)")
print(f"\n  → Good news: no extreme multicollinearity. All 4 features contribute unique information.")


Key observations:
  • PM2.5 ↔ Temperature: r = -0.326 (negative — polluted sites tend to be cooler)
  • WBGT ↔ Humidity: r = 0.552 (positive — WBGT incorporates humidity)
  • PM2.5 ↔ WBGT: r = -0.001 (near zero — independent dimensions!)

  → Good news: no extreme multicollinearity. All 4 features contribute unique information.


---

# Step 2 — Contextual Visuals: Why Clustering?

Before we run k-means, let's *see* the data and build intuition. Good data scientists don't just run algorithms — they first ask: **"Can I already see natural groupings?"**

We'll create three key visualizations:
1. **Quadrant scatter plot** — PM2.5 vs Temperature with mean-based quadrant lines
2. **Multi-feature radar chart** — each site's environmental profile at a glance
3. **Parallel coordinates** — all 4 dimensions simultaneously

### 2.1 — Quadrant Analysis: PM2.5 vs Temperature

This is the most intuitive way to visualize sites before clustering. We draw lines at the **overall mean** of PM2.5 and temperature to create four quadrants. Each quadrant tells a story:

| Quadrant | Meaning | Environmental Implication |
|----------|---------|--------------------------|
| Top-Right | High PM2.5 + Warm | Worst compound exposure |
| Top-Left | High PM2.5 + Cool | Polluted but thermally comfortable |
| Bottom-Right | Low PM2.5 + Warm | Clean air but heat-stressed |
| Bottom-Left | Low PM2.5 + Cool | Best overall environment |

> **💡 Teaching Moment:** Quadrant analysis is a simple but powerful technique for presenting 2D data. It transforms a scatter plot into an actionable framework that non-technical stakeholders (city officials, community members) can immediately understand.

In [21]:
# Quadrant scatter: PM2.5 vs Temperature
pm25_avg = site_profiles['pm25_mean'].mean()
temp_avg = site_profiles['temp_mean'].mean()

fig = go.Figure()

# Scatter points with site labels
fig.add_trace(go.Scatter(
    x=site_profiles['temp_mean'], y=site_profiles['pm25_mean'],
    mode='markers+text',
    text=site_profiles['site_name'],
    textposition='top center',
    textfont=dict(size=9),
    marker=dict(size=12, color=site_profiles['humid_mean'],
                colorscale='Viridis', showscale=True,
                colorbar=dict(title='Humidity %')),
    hovertemplate='<b>%{text}</b><br>Temp: %{x:.1f}°F<br>PM2.5: %{y:.1f} µg/m³<extra></extra>'
))

# Quadrant dividers at overall means
fig.add_hline(y=pm25_avg, line_dash='dash', line_color='gray', line_width=1,
              annotation_text=f'Mean PM2.5 = {pm25_avg:.1f}', annotation_position='top left')
fig.add_vline(x=temp_avg, line_dash='dash', line_color='gray', line_width=1,
              annotation_text=f'Mean Temp = {temp_avg:.1f}°F', annotation_position='top right')

# Quadrant labels
fig.add_annotation(x=73.7, y=11.0, text='<b>High Pollution<br>Cool</b>',
                   showarrow=False, font=dict(size=10, color='#6f070f'), opacity=0.5)
fig.add_annotation(x=75.3, y=11.0, text='<b>High Pollution<br>Warm</b>',
                   showarrow=False, font=dict(size=10, color='#b71c1c'), opacity=0.5)
fig.add_annotation(x=73.7, y=7.7, text='<b>Low Pollution<br>Cool</b>',
                   showarrow=False, font=dict(size=10, color='#2e7d32'), opacity=0.5)
fig.add_annotation(x=75.3, y=7.7, text='<b>Low Pollution<br>Warm</b>',
                   showarrow=False, font=dict(size=10, color='#e65100'), opacity=0.5)

fig.update_layout(
    title='Site Environmental Profiles: PM2.5 vs Temperature<br><sub>Color = humidity | Dashed lines = overall means (quadrant boundaries)</sub>',
    xaxis_title='Mean Temperature (°F)',
    yaxis_title='Mean PM2.5 (µg/m³)',
    template='plotly_white',
    width=750, height=550,
    showlegend=False
)
fig.show()

# Print quadrant assignments
print("\n📊 Quadrant Assignments:")
for _, row in site_profiles.iterrows():
    q_pm = 'High' if row['pm25_mean'] > pm25_avg else 'Low'
    q_temp = 'Warm' if row['temp_mean'] > temp_avg else 'Cool'
    print(f"  {row['site_name']:18s} → {q_pm} PM2.5 / {q_temp}  ({row['pm25_mean']:.1f} µg/m³, {row['temp_mean']:.1f}°F)")


📊 Quadrant Assignments:
  Berkeley Garden    → High PM2.5 / Cool  (9.5 µg/m³, 74.4°F)
  Castle Square      → Low PM2.5 / Warm  (8.2 µg/m³, 75.3°F)
  Chin Park          → High PM2.5 / Warm  (10.5 µg/m³, 75.0°F)
  Dewey Square       → High PM2.5 / Warm  (9.7 µg/m³, 74.6°F)
  Eliot Norton       → Low PM2.5 / Cool  (9.3 µg/m³, 73.9°F)
  One Greenway       → High PM2.5 / Warm  (10.7 µg/m³, 74.5°F)
  Lyndboro Park      → High PM2.5 / Cool  (10.7 µg/m³, 74.4°F)
  Mary Soo Hoo       → Low PM2.5 / Cool  (9.1 µg/m³, 73.9°F)
  Oxford Plaza       → Low PM2.5 / Warm  (7.9 µg/m³, 75.0°F)
  Reggie Wong        → Low PM2.5 / Warm  (8.3 µg/m³, 74.7°F)
  Tai Tung Park      → Low PM2.5 / Cool  (9.4 µg/m³, 74.3°F)
  Tufts Garden       → High PM2.5 / Cool  (10.0 µg/m³, 74.0°F)


**Observations from the quadrant analysis:**

- **Top-left cluster** (High PM2.5, Cool): Tufts Garden, Lyndboro Park, One Greenway — these are the most polluted sites, likely due to proximity to major roads (Rose Kennedy Greenway corridor, I-93)
- **Top-right** (High PM2.5, Warm): Chin Park, Dewey Square — compound exposure risk (both heat and pollution)
- **Bottom-right** (Low PM2.5, Warm): Castle Square, Oxford Plaza — warm due to impervious surfaces, but cleaner air
- **Bottom-left** (Low PM2.5, Cool): Eliot Norton, Mary Soo Hoo — the most comfortable sites environmentally

Notice how the color gradient (humidity) adds a third dimension — Tufts Garden stands out as both high-pollution AND high-humidity, making it the most uncomfortable site overall.

> **Key insight:** The quadrant boundaries already suggest natural groupings. K-means will formalize this intuition, but now we can check whether the algorithm agrees with our visual assessment.

### 2.2 — Radar Chart: Multi-Dimensional Site Profiles

Radar charts (spider plots) let us see all four features at once for each site. This makes it easy to spot which sites have "similar shapes" — exactly what clustering will quantify.

> **💡 Teaching Moment:** Radar charts are great for comparing a small number of entities across multiple standardized dimensions. The key word is **standardized** — we need to normalize the features so they're on the same scale, otherwise the feature with the largest range dominates the visual.

In [22]:
# Radar chart — normalize features to 0–1 for visual comparison
from sklearn.preprocessing import MinMaxScaler

radar_features = ['pm25_mean', 'temp_mean', 'wbgt_mean', 'humid_mean']
radar_labels = ['PM2.5', 'Temperature', 'WBGT', 'Humidity']

scaler_mm = MinMaxScaler()
normalized = pd.DataFrame(
    scaler_mm.fit_transform(site_profiles[radar_features]),
    columns=radar_labels,
    index=site_profiles.index
)

# Color palette for 12 sites
colors = px.colors.qualitative.Set3[:12]

fig = go.Figure()
for i, (site_id, row) in enumerate(normalized.iterrows()):
    values = row.tolist() + [row.tolist()[0]]  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels + [radar_labels[0]],
        fill='toself',
        name=site_names[site_id],
        opacity=0.3,
        line=dict(color=colors[i], width=2)
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Site Environmental Profiles (Normalized 0–1)<br><sub>Similar shapes = similar sites → clustering candidates</sub>',
    template='plotly_white',
    width=700, height=550,
    showlegend=True,
    legend=dict(font=dict(size=9), x=1.05)
)
fig.show()

print("🔍 Notice how the radar shapes naturally group:")
print("   • One Greenway, Lyndboro, Chin Park → large PM2.5 spikes")
print("   • Oxford, Reggie Wong, Mary Soo Hoo → compact, low profiles")
print("   • Castle Square → unique shape (high temp, high WBGT, low PM2.5)")

🔍 Notice how the radar shapes naturally group:
   • One Greenway, Lyndboro, Chin Park → large PM2.5 spikes
   • Oxford, Reggie Wong, Mary Soo Hoo → compact, low profiles
   • Castle Square → unique shape (high temp, high WBGT, low PM2.5)


### 2.3 — Parallel Coordinates: All Four Dimensions at Once

Parallel coordinates are another way to see multi-dimensional patterns. Each vertical axis is one feature, and each line is one site. Sites that follow similar paths through the axes are natural cluster candidates.

In [23]:
# Parallel coordinates plot
plot_df = site_profiles[['site_name', 'pm25_mean', 'temp_mean', 'wbgt_mean', 'humid_mean']].copy()
plot_df['pm25_rank'] = plot_df['pm25_mean'].rank()  # use rank for color scale

fig = px.parallel_coordinates(
    plot_df,
    dimensions=['pm25_mean', 'temp_mean', 'wbgt_mean', 'humid_mean'],
    color='pm25_rank',
    labels={
        'pm25_mean': 'PM2.5 (µg/m³)',
        'temp_mean': 'Temperature (°F)',
        'wbgt_mean': 'WBGT (°F)',
        'humid_mean': 'Humidity (%)',
    },
    color_continuous_scale='RdYlGn_r',
    title='Parallel Coordinates: 12 Sites Across 4 Environmental Dimensions<br><sub>Red = high PM2.5 sites | Green = low PM2.5 sites | Lines with similar paths → natural clusters</sub>'
)
fig.update_layout(width=800, height=450, template='plotly_white')
fig.show()

print("🔍 Visual insight: You can see 2–3 'bundles' of lines taking similar paths.")
print("   This is exactly what k-means will detect mathematically.")

🔍 Visual insight: You can see 2–3 'bundles' of lines taking similar paths.
   This is exactly what k-means will detect mathematically.


---

# Step 3 — Didactic K-Means Implementation

Now we implement k-means clustering step by step. The algorithm is conceptually simple:

1. **Choose k** (number of clusters)
2. **Initialize** k random centroids
3. **Assign** each point to the nearest centroid
4. **Update** centroids as the mean of their assigned points
5. **Repeat** steps 3–4 until convergence

But before we can run k-means, we need to answer two critical questions:
- **What value of k?** → Elbow method + Silhouette analysis
- **Are features on the same scale?** → Standardization with StandardScaler

> **💡 Teaching Moment:** K-means uses Euclidean distance. If PM2.5 ranges from 7–11 but temperature ranges from 73–76, temperature differences will be dwarfed by PM2.5 differences. **Always standardize first.**

### 3.1 — Feature Selection & Standardization

We select our 4 features and standardize them using `StandardScaler`, which transforms each feature to have **mean = 0** and **standard deviation = 1**.

In [24]:
# Step 3.1: Feature selection and standardization
features = ['pm25_mean', 'temp_mean', 'wbgt_mean', 'humid_mean']
X_raw = site_profiles[features].values

print("BEFORE standardization:")
for i, name in enumerate(['PM2.5', 'Temp', 'WBGT', 'Humid']):
    col = X_raw[:, i]
    print(f"  {name:6s} range: {col.min():.1f} – {col.max():.1f}  (spread: {col.max() - col.min():.1f})")

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("\nAFTER standardization (mean ≈ 0, std ≈ 1):")
for i, feat in enumerate(features):
    print(f"  {feat:12s}: mean={X_scaled[:,i].mean():.4f}, std={X_scaled[:,i].std():.4f}, "
          f"range=[{X_scaled[:,i].min():.2f}, {X_scaled[:,i].max():.2f}]")

print("\n✅ Now all features contribute equally to distance calculations.")

BEFORE standardization:
  PM2.5  range: 7.9 – 10.7  (spread: 2.8)
  Temp   range: 73.9 – 75.3  (spread: 1.4)
  WBGT   range: 65.1 – 66.8  (spread: 1.7)
  Humid  range: 63.1 – 68.5  (spread: 5.4)

AFTER standardization (mean ≈ 0, std ≈ 1):
  pm25_mean   : mean=-0.0000, std=1.0000, range=[-1.66, 1.39]
  temp_mean   : mean=-0.0000, std=1.0000, range=[-1.38, 1.91]
  wbgt_mean   : mean=-0.0000, std=1.0000, range=[-1.95, 2.27]
  humid_mean  : mean=0.0000, std=1.0000, range=[-1.97, 1.91]

✅ Now all features contribute equally to distance calculations.


### 3.2 — Choosing k: The Elbow Method

The **elbow method** runs k-means for k = 2, 3, 4, ..., and plots the **inertia** (sum of squared distances from each point to its cluster center). We look for the "elbow" — the point where adding more clusters gives diminishing returns.

> **💡 Teaching Moment:** Inertia always decreases as k increases (at k = n, inertia = 0). The elbow is where the *rate of decrease* slows sharply. There's no single "correct" answer — it's a judgment call guided by domain knowledge.

In [25]:
# Step 3.2: Elbow method
k_range = range(2, 8)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouettes.append(sil)
    print(f"k={k}: Inertia={km.inertia_:.2f}, Silhouette={sil:.3f}")

# Plot elbow curve
fig = make_subplots(rows=1, cols=2, subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score'])

fig.add_trace(go.Scatter(
    x=list(k_range), y=inertias,
    mode='lines+markers', marker=dict(size=10, color='#6f070f'),
    line=dict(width=2.5), name='Inertia'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(k_range), y=silhouettes,
    mode='lines+markers', marker=dict(size=10, color='#003e2f'),
    line=dict(width=2.5), name='Silhouette'
), row=1, col=2)

# Highlight k=3
fig.add_vline(x=3, line_dash='dot', line_color='#87512d', line_width=2, row=1, col=1,
              annotation_text='k=3', annotation_position='top')
fig.add_vline(x=3, line_dash='dot', line_color='#87512d', line_width=2, row=1, col=2,
              annotation_text='k=3', annotation_position='top')

fig.update_xaxes(title_text='Number of Clusters (k)', dtick=1)
fig.update_yaxes(title_text='Inertia', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
fig.update_layout(
    title='How Many Clusters? Elbow + Silhouette Analysis',
    template='plotly_white', width=850, height=400, showlegend=False
)
fig.show()

print("\n📊 Interpretation:")
print("  • The elbow plot shows a moderate bend around k=3")
print("  • Silhouette scores increase through k=5, but k=3 is a reasonable starting point")
print("  • With only 12 data points, more than 3–4 clusters would over-segment the data")
print("  → We'll proceed with k=3 (interpretable and defensible)")

k=2: Inertia=32.00, Silhouette=0.275
k=3: Inertia=21.79, Silhouette=0.289
k=4: Inertia=14.37, Silhouette=0.327
k=5: Inertia=7.70, Silhouette=0.344
k=6: Inertia=5.32, Silhouette=0.311
k=7: Inertia=3.44, Silhouette=0.312



📊 Interpretation:
  • The elbow plot shows a moderate bend around k=3
  • Silhouette scores increase through k=5, but k=3 is a reasonable starting point
  • With only 12 data points, more than 3–4 clusters would over-segment the data
  → We'll proceed with k=3 (interpretable and defensible)


### 3.3 — Silhouette Analysis (Deep Dive)

The **silhouette score** measures how well each point fits within its assigned cluster vs. the nearest neighboring cluster. Scores range from -1 (wrong cluster) to +1 (perfectly clustered).

> **💡 Teaching Moment:** The overall silhouette score is an average. A per-sample silhouette plot shows which individual points are well-clustered and which are borderline. This is especially important with small datasets like ours (n=12).

In [26]:
# Step 3.3: Per-sample silhouette analysis for k=3
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_3 = km3.fit_predict(X_scaled)
sil_samples = silhouette_samples(X_scaled, labels_3)
avg_sil = silhouette_score(X_scaled, labels_3)

# Build a DataFrame for plotting
sil_df = pd.DataFrame({
    'site': site_profiles['site_name'].values,
    'cluster': labels_3,
    'silhouette': sil_samples
}).sort_values(['cluster', 'silhouette'], ascending=[True, False])

# Color map for clusters (matched to emoji labels: 🟡 🔴 🟢)
cluster_colors = {0: '#DAA520', 1: '#C62828', 2: '#2E7D32'}
cluster_labels = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

fig = go.Figure()
for cl in sorted(sil_df['cluster'].unique()):
    subset = sil_df[sil_df['cluster'] == cl]
    fig.add_trace(go.Bar(
        y=subset['site'], x=subset['silhouette'],
        orientation='h', name=cluster_labels[cl],
        marker_color=cluster_colors[cl],
        text=subset['silhouette'].round(3),
        textposition='outside'
    ))

fig.add_vline(x=avg_sil, line_dash='dash', line_color='gray',
              annotation_text=f'Average = {avg_sil:.3f}', annotation_position='top')

fig.update_layout(
    title=f'Per-Site Silhouette Scores (k=3, avg={avg_sil:.3f})',
    xaxis_title='Silhouette Score',
    template='plotly_white', width=700, height=450,
    barmode='relative',
    yaxis=dict(categoryorder='array', categoryarray=sil_df['site'].tolist()[::-1])
)
fig.show()

print(f"\nOverall silhouette score: {avg_sil:.3f}")
print("  → Moderate clustering quality (typical for small n with continuous features)")
print("  → All scores > 0: no site is assigned to the wrong cluster")


Overall silhouette score: 0.289
  → Moderate clustering quality (typical for small n with continuous features)
  → All scores > 0: no site is assigned to the wrong cluster


### 3.4 — Fit K-Means (k=3) & Interpret Cluster Centers

Now we fit the final model and examine what each cluster represents. The **cluster centers** (centroids) tell us the "average environmental profile" of each group.

In [27]:
# Step 3.4: Final k=3 model — assign clusters and interpret
site_profiles['cluster'] = labels_3

# Convert cluster centers back to original scale
centers_original = scaler.inverse_transform(km3.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=['PM2.5 (µg/m³)', 'Temp (°F)', 'WBGT (°F)', 'Humidity (%)'])
centers_df.index.name = 'Cluster'

print("═══ Cluster Centers (Original Scale) ═══\n")
print(centers_df.round(2).to_string())

# Name the clusters based on their environmental profile
cluster_names = {}
for i, row in centers_df.iterrows():
    if row['PM2.5 (µg/m³)'] > pm25_avg and row['Humidity (%)'] > 66:
        cluster_names[i] = '🔴 High Pollution + Humid'
    elif row['PM2.5 (µg/m³)'] < pm25_avg and row['Humidity (%)'] < 65:
        cluster_names[i] = '🟢 Cleaner & Drier'
    elif row['Temp (°F)'] > 75:
        cluster_names[i] = '🟡 Urban Heat Island'
    else:
        cluster_names[i] = '🟠 Moderate'

print("\n\n═══ Cluster Assignments ═══\n")
for cl in sorted(site_profiles['cluster'].unique()):
    sites = site_profiles[site_profiles['cluster'] == cl]
    name = cluster_names.get(cl, f'Cluster {cl}')
    print(f"{name} (Cluster {cl}):")
    for _, row in sites.iterrows():
        print(f"  • {row['site_name']:18s} — PM2.5={row['pm25_mean']:.1f}, "
              f"Temp={row['temp_mean']:.1f}°F, Humid={row['humid_mean']:.1f}%")
    print()

═══ Cluster Centers (Original Scale) ═══

         PM2.5 (µg/m³)  Temp (°F)  WBGT (°F)  Humidity (%)
Cluster                                                   
0                 8.17      75.31      66.76         66.81
1                 9.98      74.39      65.90         66.34
2                 8.45      74.51      65.45         64.25


═══ Cluster Assignments ═══

🟡 Urban Heat Island (Cluster 0):
  • Castle Square      — PM2.5=8.2, Temp=75.3°F, Humid=66.8%

🔴 High Pollution + Humid (Cluster 1):
  • Berkeley Garden    — PM2.5=9.5, Temp=74.4°F, Humid=67.2%
  • Chin Park          — PM2.5=10.5, Temp=75.0°F, Humid=64.3%
  • Dewey Square       — PM2.5=9.7, Temp=74.6°F, Humid=65.7%
  • Eliot Norton       — PM2.5=9.3, Temp=73.9°F, Humid=66.1%
  • One Greenway       — PM2.5=10.7, Temp=74.5°F, Humid=65.7%
  • Lyndboro Park      — PM2.5=10.7, Temp=74.4°F, Humid=66.3%
  • Tai Tung Park      — PM2.5=9.4, Temp=74.3°F, Humid=67.0%
  • Tufts Garden       — PM2.5=10.0, Temp=74.0°F, Humid=68.5%

🟢 Clea

### 3.5 — Visualize Clusters: 2D Scatter with PCA

Since our data has 4 dimensions, we use **Principal Component Analysis (PCA)** to project it onto 2 dimensions for visualization. PCA finds the two directions that capture the most variance.

> **💡 Teaching Moment:** PCA is a dimensionality reduction technique — it doesn't change the clustering results, it just helps us *see* them. The percentage shown on each axis tells you how much information that axis captures.

In [28]:
# Step 3.5: PCA projection and cluster visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

site_profiles['pca1'] = X_pca[:, 0]
site_profiles['pca2'] = X_pca[:, 1]

# Project cluster centers to PCA space
centers_pca = pca.transform(km3.cluster_centers_)

# Build scatter plot — colors matched to emoji labels: 🟡 🔴 🟢
color_map = {0: '#DAA520', 1: '#C62828', 2: '#2E7D32'}
fig = go.Figure()

for cl in sorted(site_profiles['cluster'].unique()):
    subset = site_profiles[site_profiles['cluster'] == cl]
    name = cluster_names.get(cl, f'Cluster {cl}')
    fig.add_trace(go.Scatter(
        x=subset['pca1'], y=subset['pca2'],
        mode='markers+text',
        text=subset['site_name'],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(size=14, color=color_map[cl], opacity=0.8,
                    line=dict(width=2, color='white')),
        name=name,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
    ))

# Plot centroids
for i, (cx, cy) in enumerate(centers_pca):
    fig.add_trace(go.Scatter(
        x=[cx], y=[cy], mode='markers',
        marker=dict(size=18, color=color_map[i], symbol='x', line=dict(width=3)),
        name=f'Centroid {i}', showlegend=False
    ))

fig.update_layout(
    title=f'K-Means Clusters (k=3) — PCA Projection<br><sub>PC1 explains {pca.explained_variance_ratio_[0]:.0%} | PC2 explains {pca.explained_variance_ratio_[1]:.0%} | Total: {pca.explained_variance_ratio_.sum():.0%} of variance</sub>',
    xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)',
    yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)',
    template='plotly_white', width=750, height=550,
)
fig.show()

print(f"PCA captures {pca.explained_variance_ratio_.sum():.0%} of the total variance in 2 dimensions.")
print(f"\nPCA loadings (what each PC emphasizes):")
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=features)
print(loadings.round(3).to_string())

PCA captures 82% of the total variance in 2 dimensions.

PCA loadings (what each PC emphasizes):
              PC1    PC2
pm25_mean  -0.546  0.088
temp_mean   0.659  0.323
wbgt_mean   0.135  0.773
humid_mean -0.499  0.540


### 3.6 — Cluster Profiles: Radar Chart Comparison

Now let's see what each cluster "looks like" across all 4 features using a radar chart. This is the most powerful way to interpret the clusters — each cluster's shape tells a different environmental story.

In [29]:
# Step 3.6: Cluster profile radar chart
radar_labels = ['PM2.5', 'Temperature', 'WBGT', 'Humidity']

# Normalize cluster centers to 0–1 for radar comparison
centers_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(centers_original),
    columns=radar_labels
)

fig = go.Figure()
for i in range(3):
    values = centers_norm.iloc[i].tolist() + [centers_norm.iloc[i].tolist()[0]]
    name = cluster_names.get(i, f'Cluster {i}')
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels + [radar_labels[0]],
        fill='toself',
        name=name,
        line=dict(color=list(color_map.values())[i], width=3),
        opacity=0.4
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Cluster Environmental Profiles (Normalized)<br><sub>Each cluster has a distinct "shape" = distinct environmental character</sub>',
    template='plotly_white', width=650, height=500,
)
fig.show()

print("🔍 Cluster Interpretations:")
for i, name in cluster_names.items():
    c = centers_original[i]
    print(f"\n  {name}:")
    print(f"    PM2.5 = {c[0]:.1f} µg/m³ | Temp = {c[1]:.1f}°F | WBGT = {c[2]:.1f}°F | Humidity = {c[3]:.1f}%")

🔍 Cluster Interpretations:

  🟡 Urban Heat Island:
    PM2.5 = 8.2 µg/m³ | Temp = 75.3°F | WBGT = 66.8°F | Humidity = 66.8%

  🔴 High Pollution + Humid:
    PM2.5 = 10.0 µg/m³ | Temp = 74.4°F | WBGT = 65.9°F | Humidity = 66.3%

  🟢 Cleaner & Drier:
    PM2.5 = 8.4 µg/m³ | Temp = 74.5°F | WBGT = 65.5°F | Humidity = 64.3%


### 3.7 — Clusters on the Original Quadrant Plot

Let's revisit our PM2.5 vs Temperature scatter plot from Step 2, but now colored by cluster assignment. This shows how the k-means algorithm's groupings compare to our earlier visual intuition.

In [30]:
# Step 3.7: Clusters overlaid on the PM2.5 vs Temperature quadrant plot
fig = go.Figure()

for cl in sorted(site_profiles['cluster'].unique()):
    subset = site_profiles[site_profiles['cluster'] == cl]
    name = cluster_names.get(cl, f'Cluster {cl}')
    fig.add_trace(go.Scatter(
        x=subset['temp_mean'], y=subset['pm25_mean'],
        mode='markers+text',
        text=subset['site_name'],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(size=14, color=color_map[cl], opacity=0.85,
                    line=dict(width=2, color='white')),
        name=name,
        hovertemplate='<b>%{text}</b><br>Temp: %{x:.1f}°F<br>PM2.5: %{y:.1f} µg/m³<extra></extra>'
    ))

# Quadrant dividers
fig.add_hline(y=pm25_avg, line_dash='dash', line_color='gray', line_width=1, opacity=0.5)
fig.add_vline(x=temp_avg, line_dash='dash', line_color='gray', line_width=1, opacity=0.5)

# Quadrant labels (subtle)
fig.add_annotation(x=73.7, y=11.0, text='High Pollution / Cool', showarrow=False,
                   font=dict(size=9, color='gray'), opacity=0.4)
fig.add_annotation(x=75.3, y=11.0, text='High Pollution / Warm', showarrow=False,
                   font=dict(size=9, color='gray'), opacity=0.4)
fig.add_annotation(x=73.7, y=7.7, text='Low Pollution / Cool', showarrow=False,
                   font=dict(size=9, color='gray'), opacity=0.4)
fig.add_annotation(x=75.3, y=7.7, text='Low Pollution / Warm', showarrow=False,
                   font=dict(size=9, color='gray'), opacity=0.4)

fig.update_layout(
    title='K-Means Clusters (k=3) on PM2.5 × Temperature Space<br><sub>Clusters consider all 4 features, not just these 2 — that\'s why boundaries aren\'t perfectly aligned with quadrants</sub>',
    xaxis_title='Mean Temperature (°F)',
    yaxis_title='Mean PM2.5 (µg/m³)',
    template='plotly_white', width=750, height=550,
)
fig.show()

print("🔍 Key observation: K-means clusters DON'T perfectly match the quadrant grid.")
print("   That's because the algorithm uses ALL 4 features (including WBGT and humidity),")
print("   not just PM2.5 and temperature. This is the power of multi-dimensional clustering!")
print("   It captures patterns that simple 2D visual inspection would miss.")

🔍 Key observation: K-means clusters DON'T perfectly match the quadrant grid.
   That's because the algorithm uses ALL 4 features (including WBGT and humidity),
   not just PM2.5 and temperature. This is the power of multi-dimensional clustering!
   It captures patterns that simple 2D visual inspection would miss.


### 3.7b — Adjusting the Quadrant Boundaries to Match Cluster Structure

The chart above uses the **arithmetic mean** of all 12 sites as the quadrant dividers — the same naive boundaries we used in Step 2 before running k-means. But now that we *have* cluster results, we can ask: **where should the boundaries actually be to separate the clusters in this 2D projection?**

By shifting the lines to **PM2.5 = 9.2 µg/m³** and **Temperature = 75.25°F**, the quadrants align cleanly with the three clusters. Why do these values work?

- **PM2.5 = 9.2** sits between the 🟢 Cleaner cluster center (~8.4) and the 🔴 High Pollution center (~10.0) — it's approximately the **midpoint between centroids** on the PM2.5 axis
- **Temp = 75.25°F** isolates Castle Square (🟡 Urban Heat Island at 75.3°F) from the rest of the sites, which all fall below 75°F

> **💡 Teaching Moment:** The original mean-based boundaries are *descriptive* (they split "above average" vs "below average"). The adjusted boundaries are *cluster-informed* — they reflect the actual decision surface between groups. This is a key insight: **clustering doesn't just label points, it also reveals where the natural boundaries lie.** When communicating results to stakeholders, using cluster-informed boundaries makes the visualization match the analysis.

In [31]:
# Step 3.7b: Cluster-informed quadrant boundaries
pm25_boundary = 9.2    # midpoint between Cleaner (~8.4) and High Pollution (~10.0)
temp_boundary = 75.25  # isolates Castle Square (75.3°F) from the rest (<75°F)

fig = go.Figure()

for cl in sorted(site_profiles['cluster'].unique()):
    subset = site_profiles[site_profiles['cluster'] == cl]
    name = cluster_names.get(cl, f'Cluster {cl}')
    fig.add_trace(go.Scatter(
        x=subset['temp_mean'], y=subset['pm25_mean'],
        mode='markers+text',
        text=subset['site_name'],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(size=14, color=color_map[cl], opacity=0.85,
                    line=dict(width=2, color='white')),
        name=name,
        hovertemplate='<b>%{text}</b><br>Temp: %{x:.1f}°F<br>PM2.5: %{y:.1f} µg/m³<extra></extra>'
    ))

# Cluster-informed quadrant dividers
fig.add_hline(y=pm25_boundary, line_dash='dash', line_color='#555', line_width=1.5,
              annotation_text=f'PM2.5 = {pm25_boundary}', annotation_position='top left')
fig.add_vline(x=temp_boundary, line_dash='dash', line_color='#555', line_width=1.5,
              annotation_text=f'Temp = {temp_boundary}°F', annotation_position='top right')

# Quadrant labels matching cluster identities
fig.add_annotation(x=73.7, y=11.0, text='<b>🔴 High Pollution<br>+ Humid</b>',
                   showarrow=False, font=dict(size=10, color='#C62828'), opacity=0.6)
fig.add_annotation(x=75.6, y=11.0, text='<b>(no sites)</b>',
                   showarrow=False, font=dict(size=9, color='gray'), opacity=0.3)
fig.add_annotation(x=73.7, y=7.5, text='<b>🟢 Cleaner<br>& Drier</b>',
                   showarrow=False, font=dict(size=10, color='#2E7D32'), opacity=0.6)
fig.add_annotation(x=75.6, y=7.5, text='<b>🟡 Urban Heat<br>Island</b>',
                   showarrow=False, font=dict(size=10, color='#DAA520'), opacity=0.6)

fig.update_layout(
    title='Cluster-Informed Quadrants: PM2.5 = 9.2 µg/m³ × Temp = 75.25°F<br>'
          '<sub>Boundaries shifted from simple means to cluster midpoints — now the quadrants match k-means results</sub>',
    xaxis_title='Mean Temperature (°F)',
    yaxis_title='Mean PM2.5 (µg/m³)',
    template='plotly_white', width=750, height=550,
)
fig.show()

# Compare old vs new boundaries
print("📐 Boundary comparison:")
print(f"   Original (Step 2):  PM2.5 = {pm25_avg:.1f} µg/m³  |  Temp = {temp_avg:.1f}°F  (arithmetic means)")
print(f"   Adjusted (Step 3):  PM2.5 = {pm25_boundary} µg/m³  |  Temp = {temp_boundary}°F  (cluster-informed)")
print(f"\n   Shift: PM2.5 moved {'down' if pm25_boundary < pm25_avg else 'up'} by {abs(pm25_boundary - pm25_avg):.1f} µg/m³"
      f"  |  Temp moved {'down' if temp_boundary < temp_avg else 'up'} by {abs(temp_boundary - temp_avg):.2f}°F")
print("\n✅ With these boundaries, the 3 clusters map cleanly onto the quadrant grid:")
print("   • Top-left  → 🔴 High Pollution + Humid (8 sites)")
print("   • Bottom-left → 🟢 Cleaner & Drier (3 sites)")
print("   • Bottom-right → 🟡 Urban Heat Island (Castle Square)")

📐 Boundary comparison:
   Original (Step 2):  PM2.5 = 9.4 µg/m³  |  Temp = 74.5°F  (arithmetic means)
   Adjusted (Step 3):  PM2.5 = 9.2 µg/m³  |  Temp = 75.25°F  (cluster-informed)

   Shift: PM2.5 moved down by 0.2 µg/m³  |  Temp moved up by 0.75°F

✅ With these boundaries, the 3 clusters map cleanly onto the quadrant grid:
   • Top-left  → 🔴 High Pollution + Humid (8 sites)
   • Bottom-left → 🟢 Cleaner & Drier (3 sites)
   • Bottom-right → 🟡 Urban Heat Island (Castle Square)


### 3.8 — Cluster Bar Chart: Feature Comparison

A grouped bar chart gives a clear, direct comparison of cluster centers across all features — the most accessible visualization for non-technical audiences.

In [32]:
# Step 3.8: Grouped bar chart of cluster centers
bar_features = ['PM2.5 (µg/m³)', 'Temp (°F)', 'WBGT (°F)', 'Humidity (%)']

fig = make_subplots(rows=1, cols=4, subplot_titles=bar_features, shared_yaxes=False)

for j, feat in enumerate(bar_features):
    for i in range(3):
        name = cluster_names.get(i, f'Cluster {i}')
        fig.add_trace(go.Bar(
            x=[name], y=[centers_df.iloc[i][feat]],
            marker_color=color_map[i],
            name=name, showlegend=(j == 0),
            text=[f'{centers_df.iloc[i][feat]:.1f}'],
            textposition='outside'
        ), row=1, col=j+1)

fig.update_layout(
    title='Cluster Centers: Side-by-Side Feature Comparison',
    template='plotly_white', width=950, height=400,
    barmode='group', showlegend=True,
)
fig.show()

---

## Step 3 — Synthesis: What Did Clustering Reveal?

K-means clustering with k=3 identified **three distinct environmental profiles** among Chinatown's 12 open-space sites:

| Cluster | Profile | Sites | Key Characteristic |
|---------|---------|-------|--------------------|
| 🟡 Cluster 0 | Urban Heat Island | Castle Square (1 site) | Warmest site (75.3°F), high WBGT (66.8°F), moderate PM2.5 — impervious surface-dominated microclimate |
| 🔴 Cluster 1 | Elevated Pollution | Berkeley, Chin, Dewey, Greenway, Lyndboro, Tai Tung, Eliot Norton, Tufts (8 sites) | Highest mean PM2.5 (~10 µg/m³), moderate temps, higher humidity — likely driven by road proximity |
| 🟢 Cluster 2 | Cleaner & Drier | Mary Soo Hoo, Oxford, Reggie Wong (3 sites) | Lowest PM2.5 (~8.4 µg/m³), lowest humidity (~64%), moderate temps — greater setback from major roads |

### Public Health Implications

- **Cluster 1 sites** (8 of 12) experience the highest PM2.5 and should be prioritized for air quality interventions (tree planting, traffic reduction)
- **Castle Square** (Cluster 0) is a standalone "heat island" — unique heat exposure profile suggests targeted cooling strategies (shade structures, reflective surfaces)
- **Cluster 2 sites** offer the most comfortable environmental conditions — understanding why (road setback? greenspace?) can inform future park design

> **💡 Teaching Moment:** An uneven cluster distribution (8-1-3) is not a failure — it's a finding! It tells us that most Chinatown sites share a similar pollution profile, with only a few exceptions. This is actionable information for health officials.

### Next Steps

The notebook continues with:
- **Step 5** — Exporting data for PowerBI dashboards (below)
- **Step 4** (separate) — Interactive React dashboard with filters and rich visualizations

---

## Step 5 — Data Export for PowerBI Dashboard

In this final step we export clean, analysis-ready CSV files that can be imported directly into PowerBI. Each file maps to specific dashboard visuals.

### PowerBI Dashboard Design

We recommend **two dashboard pages**:

---

#### Page 1: Cluster Overview & Profiles

| Position | Visual Type | Data Source | Description |
|----------|------------|-------------|-------------|
| **Top banner** | Card / KPI tiles (×4) | `clustering_kpi.csv` | Sites (12), Clusters (3), Silhouette (0.289), PCA Variance (82%) |
| **Left half** | Scatter Chart | `site_cluster_assignments.csv` | PM2.5 vs Temp colored by cluster, with constant reference lines at PM2.5=9.2 and Temp=75.25 |
| **Right half** | Radar Chart (or Filled Map) | `cluster_centers.csv` | Normalized 0–1 profiles per cluster across all 4 features |
| **Bottom strip** | Matrix / Table | `site_cluster_assignments.csv` | Full site detail table with conditional formatting by cluster color |

#### Page 2: Method Validation & Spatial View

| Position | Visual Type | Data Source | Description |
|----------|------------|-------------|-------------|
| **Top-left** | Line Chart | `elbow_silhouette.csv` | Dual-axis: Inertia (left) and Silhouette (right) vs k. Highlight k=3 |
| **Top-right** | Bar Chart | `site_silhouette_scores.csv` | Horizontal bars of per-site silhouette, colored by cluster |
| **Bottom-left** | Scatter Chart | `pca_projection.csv` | PCA 2D projection with cluster coloring |
| **Bottom-right** | Filled Map | `site_cluster_assignments.csv` | Lat/Lon map of Chinatown sites, markers sized by PM2.5, colored by cluster |

---

### DAX Measures (add in PowerBI)

```dax
// Cluster count
Cluster Count = DISTINCTCOUNT(site_cluster_assignments[cluster])

// Average silhouette by cluster
Avg Silhouette = AVERAGE(site_silhouette_scores[silhouette])

// PM2.5 above threshold flag
PM25 Above Threshold = IF([pm25_mean] > 9.2, "Above", "Below")

// Conditional color (use in visual formatting rules)
Cluster Color = 
    SWITCH(
        SELECTEDVALUE(site_cluster_assignments[cluster]),
        0, "#DAA520",
        1, "#C62828",
        2, "#2E7D32",
        "#87512d"
    )
```

> **💡 Teaching Moment:** When moving from Python to PowerBI, the key shift is from _code-driven_ to _drag-and-drop_ visualization. But the data preparation is identical — clean, aggregated tables with one row per entity (site, cluster, k-value). That's exactly what we export below.

### Export 1: Site Cluster Assignments

The core table — one row per site with environmental means, cluster assignment, coordinates, and photo path. Powers the scatter chart, map, and detail table in PowerBI.

In [33]:
# Export 1: Site cluster assignments with coordinates
from pathlib import Path

EXPORT_DIR = Path('../data/powerbi')
EXPORT_DIR.mkdir(exist_ok=True)

SITE_COORDS = {
    "berkley": (42.34483, -71.06857), "castle": (42.3440, -71.0663),
    "chin": (42.3512, -71.0595), "dewey": (42.3534, -71.0551),
    "eliotnorton": (42.3509, -71.0644), "greenway": (42.35012, -71.06012),
    "lyndenboro": (42.35001, -71.06614), "msh": (42.35129, -71.05997),
    "oxford": (42.35252, -71.06107), "reggie": (42.3497, -71.0609),
    "taitung": (42.34901, -71.06192), "tufts": (42.3474, -71.0656),
}

export_sites = site_profiles.copy()
export_sites['cluster'] = labels_3
export_sites['cluster_name'] = export_sites['cluster'].map(cluster_names)
export_sites['cluster_color'] = export_sites['cluster'].map(color_map)
export_sites['cluster_emoji'] = export_sites['cluster'].map({
    0: '🟡', 1: '🔴', 2: '🟢'
})
export_sites['site_name'] = export_sites.index.map(site_names)
export_sites['lat'] = export_sites.index.map(lambda s: SITE_COORDS[s][0])
export_sites['lon'] = export_sites.index.map(lambda s: SITE_COORDS[s][1])
export_sites['pca1'] = X_pca[:, 0].round(3)
export_sites['pca2'] = X_pca[:, 1].round(3)
export_sites['silhouette'] = sil_samples.round(3)

export_sites.to_csv(EXPORT_DIR / 'site_cluster_assignments.csv')
print(f"✅ Exported site_cluster_assignments.csv ({len(export_sites)} rows × {export_sites.shape[1]} cols)")
display(export_sites.head())

✅ Exported site_cluster_assignments.csv (12 rows × 16 cols)


,pm25_mean,temp_mean,wbgt_mean,humid_mean,pm25_std,n_obs,site_name,cluster,pca1,pca2,cluster_name,cluster_color,cluster_emoji,lat,lon,silhouette
site_id,,,,,,,,,,,,,,,,
berkley,9.53,74.41,66.10,67.18,4.94,2445,Berkeley Garden,1,-0.577,0.916,🔴 High Pollution + Humid,#C62828,🔴,42.34483,-71.06857,0.465
castle,8.17,75.31,66.76,66.81,5.42,3793,Castle Square,0,1.992,2.613,🟡 Urban Heat Island,#DAA520,🟡,42.34400,-71.06630,0.000
chin,10.49,75.02,66.01,64.28,5.93,2199,Chin Park,1,0.794,0.181,🔴 High Pollution + Humid,#C62828,🔴,42.35120,-71.05950,0.176
dewey,9.69,74.57,65.92,65.74,5.50,4889,Dewey Square,1,0.026,0.148,🔴 High Pollution + Humid,#C62828,🔴,42.35340,-71.05510,0.368
eliotnorton,9.29,73.93,65.48,66.09,4.89,3888,Eliot Norton,1,-1.002,-1.101,🔴 High Pollution + Humid,#C62828,🔴,42.35090,-71.06440,0.052


### Export 2: Cluster Centers

One row per cluster with original-scale and normalized (0–1) feature values. Powers the radar chart and cluster profile cards in PowerBI.

In [34]:
# Export 2: Cluster centers (original and normalized)
from sklearn.preprocessing import MinMaxScaler

export_centers = centers_df.copy()
export_centers['cluster'] = range(3)
export_centers['cluster_name'] = [cluster_names[i] for i in range(3)]
export_centers['cluster_color'] = [color_map[i] for i in range(3)]
export_centers['n_sites'] = [int((labels_3 == i).sum()) for i in range(3)]
export_centers['sites'] = [
    ', '.join(site_profiles.index[labels_3 == i].tolist()) for i in range(3)
]

# Add normalized values (0-1) for radar chart
scaler_mm = MinMaxScaler()
norm_vals = scaler_mm.fit_transform(centers_original)
for j, feat in enumerate(['pm25_norm', 'temp_norm', 'wbgt_norm', 'humidity_norm']):
    export_centers[feat] = norm_vals[:, j].round(3)

export_centers.to_csv(EXPORT_DIR / 'cluster_centers.csv', index=False)
print(f"✅ Exported cluster_centers.csv ({len(export_centers)} rows × {export_centers.shape[1]} cols)")
display(export_centers)

✅ Exported cluster_centers.csv (3 rows × 13 cols)


,PM2.5 (µg/m³),Temp (°F),WBGT (°F),Humidity (%),cluster,cluster_name,cluster_color,n_sites,sites,pm25_norm,temp_norm,wbgt_norm,humidity_norm
Cluster,,,,,,,,,,,,,
0,8.170000,75.310000,66.760000,66.810000,0,🟡 Urban Heat Island,#DAA520,1,castle,0.000,1.000,1.000,1.000
1,9.975000,74.390000,65.902500,66.340000,1,🔴 High Pollution + Humid,#C62828,8,"berkley, chin, dewey, eliotnorton, greenway, l...",1.000,0.000,0.344,0.816
2,8.446667,74.513333,65.453333,64.253333,2,🟢 Cleaner & Drier,#2E7D32,3,"msh, oxford, reggie",0.153,0.134,0.000,0.000


### Export 3: Elbow & Silhouette Curve

One row per k (2–7) with inertia and silhouette score. Powers the dual-axis line chart in PowerBI Page 2.

In [35]:
# Export 3: Elbow and silhouette data
export_elbow = pd.DataFrame({
    'k': list(range(2, 8)),
    'inertia': [round(x, 2) for x in inertias],
    'silhouette': [round(x, 3) for x in silhouettes],
    'is_optimal': [k == 3 for k in range(2, 8)],
})

export_elbow.to_csv(EXPORT_DIR / 'elbow_silhouette.csv', index=False)
print(f"✅ Exported elbow_silhouette.csv ({len(export_elbow)} rows)")
display(export_elbow)

✅ Exported elbow_silhouette.csv (6 rows)


,k,inertia,silhouette,is_optimal
0,2,32.00,0.275,False
1,3,21.79,0.289,True
2,4,14.37,0.327,False
3,5,7.70,0.344,False
4,6,5.32,0.311,False
5,7,3.44,0.312,False


### Export 4: Per-Site Silhouette Scores

One row per site with its silhouette score and cluster assignment. Powers the horizontal bar chart in PowerBI Page 2.

In [36]:
# Export 4: Per-site silhouette scores
export_sil = pd.DataFrame({
    'site_id': site_profiles.index,
    'site_name': [site_names[s] for s in site_profiles.index],
    'cluster': labels_3,
    'cluster_name': [cluster_names[c] for c in labels_3],
    'cluster_color': [color_map[c] for c in labels_3],
    'silhouette': sil_samples.round(3),
})
export_sil = export_sil.sort_values('silhouette', ascending=False)

export_sil.to_csv(EXPORT_DIR / 'site_silhouette_scores.csv', index=False)
print(f"✅ Exported site_silhouette_scores.csv ({len(export_sil)} rows)")
display(export_sil)

✅ Exported site_silhouette_scores.csv (12 rows)


,site_id,site_name,cluster,cluster_name,cluster_color,silhouette
6,lyndenboro,Lyndboro Park,1,🔴 High Pollution + Humid,#C62828,0.509
0,berkley,Berkeley Garden,1,🔴 High Pollution + Humid,#C62828,0.465
10,taitung,Tai Tung Park,1,🔴 High Pollution + Humid,#C62828,0.444
5,greenway,One Greenway,1,🔴 High Pollution + Humid,#C62828,0.431
11,tufts,Tufts Garden,1,🔴 High Pollution + Humid,#C62828,0.404
3,dewey,Dewey Square,1,🔴 High Pollution + Humid,#C62828,0.368
8,oxford,Oxford Plaza,2,🟢 Cleaner & Drier,#2E7D32,0.358
9,reggie,Reggie Wong,2,🟢 Cleaner & Drier,#2E7D32,0.204
2,chin,Chin Park,1,🔴 High Pollution + Humid,#C62828,0.176
7,msh,Mary Soo Hoo,2,🟢 Cleaner & Drier,#2E7D32,0.061


### Export 5: PCA Projection

One row per site with PC1 and PC2 coordinates. Powers the 2D scatter chart in PowerBI Page 2. Also includes PCA loadings as a separate file for optional use.

In [37]:
# Export 5: PCA projection + loadings
export_pca = pd.DataFrame({
    'site_id': site_profiles.index,
    'site_name': [site_names[s] for s in site_profiles.index],
    'cluster': labels_3,
    'cluster_name': [cluster_names[c] for c in labels_3],
    'cluster_color': [color_map[c] for c in labels_3],
    'pca1': X_pca[:, 0].round(3),
    'pca2': X_pca[:, 1].round(3),
})

export_pca.to_csv(EXPORT_DIR / 'pca_projection.csv', index=False)
print(f"✅ Exported pca_projection.csv ({len(export_pca)} rows)")

# PCA loadings (optional — for advanced PowerBI tooltip)
feature_names = ['PM2.5', 'Temperature', 'WBGT', 'Humidity']
export_loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1_loading', 'PC2_loading'],
    index=feature_names,
)
export_loadings['variance_explained'] = ''
export_loadings.loc[feature_names[0], 'variance_explained'] = f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}%"
export_loadings.loc[feature_names[1], 'variance_explained'] = f"PC2: {pca.explained_variance_ratio_[1]*100:.1f}%"

export_loadings.to_csv(EXPORT_DIR / 'pca_loadings.csv')
print(f"✅ Exported pca_loadings.csv ({len(export_loadings)} rows)")
display(export_pca.head())
display(export_loadings)

✅ Exported pca_projection.csv (12 rows)
✅ Exported pca_loadings.csv (4 rows)


,site_id,site_name,cluster,cluster_name,cluster_color,pca1,pca2
0,berkley,Berkeley Garden,1,🔴 High Pollution + Humid,#C62828,-0.577,0.916
1,castle,Castle Square,0,🟡 Urban Heat Island,#DAA520,1.992,2.613
2,chin,Chin Park,1,🔴 High Pollution + Humid,#C62828,0.794,0.181
3,dewey,Dewey Square,1,🔴 High Pollution + Humid,#C62828,0.026,0.148
4,eliotnorton,Eliot Norton,1,🔴 High Pollution + Humid,#C62828,-1.002,-1.101


,PC1_loading,PC2_loading,variance_explained
PM2.5,-0.546265,0.088143,PC1: 41.6%
Temperature,0.658999,0.322850,PC2: 40.4%
WBGT,0.135460,0.772530,
Humidity,-0.498965,0.539626,


### Export 6: KPI Summary

A single-row summary file for the KPI card tiles at the top of the PowerBI dashboard.

In [38]:
# Export 6: KPI summary
export_kpi = pd.DataFrame([{
    'n_sites': 12,
    'n_clusters': 3,
    'silhouette_score': round(float(avg_sil), 3),
    'pca_variance_pct': round(float(pca.explained_variance_ratio_.sum() * 100), 1),
    'pm25_boundary': 9.2,
    'temp_boundary': 75.25,
    'pm25_overall_mean': round(float(site_profiles['pm25_mean'].mean()), 2),
    'temp_overall_mean': round(float(site_profiles['temp_mean'].mean()), 2),
    'study_period': 'Jul 19 – Aug 23, 2023',
    'dataset_rows': len(df),
}])

export_kpi.to_csv(EXPORT_DIR / 'clustering_kpi.csv', index=False)
print(f"✅ Exported clustering_kpi.csv")
display(export_kpi)

✅ Exported clustering_kpi.csv


,n_sites,n_clusters,silhouette_score,pca_variance_pct,pm25_boundary,temp_boundary,pm25_overall_mean,temp_overall_mean,study_period,dataset_rows
0,12,3,0.289,82.0,9.2,75.25,9.44,74.5,"Jul 19 – Aug 23, 2023",48123


In [39]:
# Summary: list all exported files
import os

print("📁 PowerBI Export Summary")
print("=" * 60)
for f in sorted(EXPORT_DIR.glob('*.csv')):
    size_kb = os.path.getsize(f) / 1024
    rows = len(pd.read_csv(f))
    print(f"  {f.name:<35} {rows:>3} rows  ({size_kb:.1f} KB)")
print("=" * 60)
print(f"\n📂 All files saved to: {EXPORT_DIR.resolve()}")
print("\n💡 To import into PowerBI:")
print("   1. Open PowerBI Desktop → Get Data → Text/CSV")
print("   2. Import each CSV file as a separate table")
print("   3. Use the cluster_color column for conditional formatting")
print("   4. Add the DAX measures from the markdown cell above")
print("   5. Build visuals following the layout guide in Step 5")

📁 PowerBI Export Summary
  cluster_centers.csv                   3 rows  (0.5 KB)
  clustering_kpi.csv                    1 rows  (0.2 KB)
  elbow_silhouette.csv                  6 rows  (0.1 KB)
  pca_loadings.csv                      4 rows  (0.2 KB)
  pca_projection.csv                   12 rows  (0.9 KB)
  site_cluster_assignments.csv         12 rows  (1.7 KB)
  site_silhouette_scores.csv           12 rows  (0.8 KB)

📂 All files saved to: /Users/joaoquintanilha/Downloads/project-hero/data/powerbi

💡 To import into PowerBI:
   1. Open PowerBI Desktop → Get Data → Text/CSV
   2. Import each CSV file as a separate table
   3. Use the cluster_color column for conditional formatting
   4. Add the DAX measures from the markdown cell above
   5. Build visuals following the layout guide in Step 5


---

### 📊 How to Build Each Chart in PowerBI

Now that the data is exported, here's a **step-by-step guide** for building each visual in PowerBI Desktop. Follow along in order — each chart builds on the previous imports.

---

#### 0. Import the Data

1. Open **PowerBI Desktop** → **Home** → **Get Data** → **Text/CSV**
2. Import each of the 7 CSV files (one at a time). For each:
   - Navigate to `data/powerbi/`
   - Select the file → **Load** (not Transform — the data is already clean)
3. After all imports, go to **Model view** (left sidebar) and verify you see 7 tables
4. No relationships are needed — each table is self-contained

> **💡 Tip:** Rename the tables by right-clicking → **Rename** to remove underscores (e.g., `Site Cluster Assignments` instead of `site_cluster_assignments`)

---

#### 1. KPI Cards (Page 1 — Top Banner)

**Visual type:** Card (×4)  
**Data source:** `clustering_kpi.csv`

| Step | Action |
|------|--------|
| 1 | Click empty canvas → **Visualizations** pane → **Card** icon |
| 2 | Drag `n_sites` into the **Fields** well → the card shows "12" |
| 3 | In **Format** pane → **Callout value** → font size 28, color `#6f070f` |
| 4 | **Category label** → turn ON → text = "Monitoring Sites" |
| 5 | Resize to a small tile (~200×100 px) and position at top-left |
| 6 | Repeat for `n_clusters` ("Clusters"), `silhouette_score` ("Silhouette Score"), `pca_variance_pct` ("PCA Variance %") |

Arrange the 4 cards in a horizontal row across the top of Page 1.

---

#### 2. Environmental Quadrant Scatter (Page 1 — Left Half)

**Visual type:** Scatter Chart  
**Data source:** `site_cluster_assignments.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Scatter Chart** visual |
| 2 | **X Axis** → drag `temp_mean` |
| 3 | **Y Axis** → drag `pm25_mean` |
| 4 | **Legend** → drag `cluster_name` (this colors points by cluster) |
| 5 | **Details** → drag `site_name` (so each dot = one site) |
| 6 | **Size** → leave blank or drag `n_obs` for bubble sizing |

**Add quadrant reference lines:**

| Step | Action |
|------|--------|
| 7 | Click the chart → **Format** → **Analytics** (magnifying glass icon) |
| 8 | **X-Axis Constant Line** → **+ Add line** → Value = `75.25`, Color = gray, Style = Dashed, Label = "Temp = 75.25°F" |
| 9 | **Y-Axis Constant Line** → **+ Add line** → Value = `9.2`, Color = gray, Style = Dashed, Label = "PM2.5 = 9.2" |

**Set cluster colors manually:**

| Step | Action |
|------|--------|
| 10 | **Format** → **Data colors** → click each cluster series |
| 11 | 🟡 Urban Heat Island → `#DAA520` |
| 12 | 🔴 High Pollution + Humid → `#C62828` |
| 13 | 🟢 Cleaner & Drier → `#2E7D32` |

**Add data labels:**

| Step | Action |
|------|--------|
| 14 | **Format** → **Data labels** → ON → choose `site_name` as label |
| 15 | Position = Above, font size 8 |

---

#### 3. Cluster Radar / Profile Chart (Page 1 — Right Half)

**Visual type:** Custom visual — **Radar Chart** from AppSource  
**Data source:** `cluster_centers.csv`

> **⚠️ Note:** PowerBI doesn't have a built-in radar chart. You need to install one from AppSource.

| Step | Action |
|------|--------|
| 1 | **Home** → **Get More Visuals** (… icon in Visualizations pane) → search "Radar Chart" |
| 2 | Install **"Radar Chart"** by MAQ Software (free) |
| 3 | Insert the Radar Chart visual |
| 4 | **Category** → drag `cluster_name` |
| 5 | **Values** → drag `pm25_norm`, `temp_norm`, `wbgt_norm`, `humidity_norm` (the 0–1 normalized columns) |
| 6 | **Format** → adjust colors to match: 🟡 `#DAA520`, 🔴 `#C62828`, 🟢 `#2E7D32` |

**Alternative (no AppSource):** Use a **Stacked Bar Chart** with normalized values as a simpler fallback:

| Step | Action |
|------|--------|
| 1 | Unpivot the 4 normalized columns in Power Query: **Transform** → **Unpivot Columns** on `pm25_norm`, `temp_norm`, `wbgt_norm`, `humidity_norm` |
| 2 | Insert a **Clustered Bar Chart** → Axis = Attribute (feature name), Values = Value, Legend = `cluster_name` |

---

#### 4. Site Detail Table (Page 1 — Bottom Strip)

**Visual type:** Table (or Matrix)  
**Data source:** `site_cluster_assignments.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Table** visual |
| 2 | Drag these columns in order: `site_name`, `cluster_emoji`, `cluster_name`, `pm25_mean`, `temp_mean`, `wbgt_mean`, `humid_mean`, `silhouette` |
| 3 | **Format** → **Column headers** → bold, background color `#6f070f`, font color white |
| 4 | **Style** → choose "Alternating rows" for readability |

**Add conditional formatting (color by cluster):**

| Step | Action |
|------|--------|
| 5 | Click the column header `cluster_name` → dropdown arrow → **Conditional formatting** → **Background color** |
| 6 | Format by → **Rules** |
| 7 | Rule 1: If value contains "Heat Island" → `#FFF3CD` (light gold) |
| 8 | Rule 2: If value contains "High Pollution" → `#FFCDD2` (light red) |
| 9 | Rule 3: If value contains "Cleaner" → `#C8E6C9` (light green) |

---

#### 5. Elbow + Silhouette Dual-Axis Line Chart (Page 2 — Top Left)

**Visual type:** Line Chart (dual Y-axis)  
**Data source:** `elbow_silhouette.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Line Chart** visual |
| 2 | **X-Axis** → drag `k` |
| 3 | **Y-Axis** → drag `inertia` |
| 4 | **Secondary Y-Axis** → drag `silhouette` |
| 5 | The chart now shows two lines with independent scales |

**Highlight k=3:**

| Step | Action |
|------|--------|
| 6 | **Format** → **Analytics** → **X-Axis Constant Line** → Value = `3`, Color = `#87512d`, Style = Dotted |
| 7 | Label = "Optimal k=3" |

**Style the lines:**

| Step | Action |
|------|--------|
| 8 | **Format** → **Lines** → Inertia: color `#6f070f`, width 3 |
| 9 | Silhouette: color `#003e2f`, width 3 |
| 10 | **Markers** → ON for both series (size 6) |

**Add title:** "How Many Clusters? Elbow Method + Silhouette Score"

---

#### 6. Per-Site Silhouette Bar Chart (Page 2 — Top Right)

**Visual type:** Bar Chart (horizontal)  
**Data source:** `site_silhouette_scores.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Clustered Bar Chart** visual |
| 2 | **Y-Axis** → drag `site_name` |
| 3 | **X-Axis** → drag `silhouette` |
| 4 | **Legend** → drag `cluster_name` |

**Add the average reference line:**

| Step | Action |
|------|--------|
| 5 | **Format** → **Analytics** → **X-Axis Constant Line** → Value = `0.289`, Color = gray, Style = Dashed |
| 6 | Label = "Average = 0.289" |

**Set cluster colors:** Same as chart 2 — `#DAA520`, `#C62828`, `#2E7D32`

**Sort:** Click the chart → **More options** (⋯) → **Sort by** → `silhouette` → **Sort descending**

---

#### 7. PCA Projection Scatter (Page 2 — Bottom Left)

**Visual type:** Scatter Chart  
**Data source:** `pca_projection.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Scatter Chart** visual |
| 2 | **X Axis** → drag `pca1` |
| 3 | **Y Axis** → drag `pca2` |
| 4 | **Legend** → drag `cluster_name` |
| 5 | **Details** → drag `site_name` |
| 6 | Set cluster colors: 🟡 `#DAA520`, 🔴 `#C62828`, 🟢 `#2E7D32` |
| 7 | **Data labels** → ON → show `site_name`, position Above |

**Axis labels:** Rename X to "PC1 (55% variance)" and Y to "PC2 (27% variance)" in **Format** → **X Axis** → **Title** → custom text.

---

#### 8. Chinatown Map (Page 2 — Bottom Right)

**Visual type:** Map (built-in) or ArcGIS Maps  
**Data source:** `site_cluster_assignments.csv`

| Step | Action |
|------|--------|
| 1 | Insert a **Map** visual (globe icon in Visualizations) |
| 2 | **Latitude** → drag `lat` |
| 3 | **Longitude** → drag `lon` |
| 4 | **Legend** → drag `cluster_name` |
| 5 | **Size** → drag `pm25_mean` (larger bubbles = more pollution) |
| 6 | **Tooltips** → drag `site_name`, `temp_mean`, `humid_mean`, `silhouette` |

**Set cluster colors:**

| Step | Action |
|------|--------|
| 7 | **Format** → **Data colors** → manually set each cluster series |
| 8 | Zoom the map to Chinatown: **Format** → **Map settings** → **Auto zoom** = ON |

> **💡 Tip:** If the built-in Map visual is too limited, try **ArcGIS Maps for Power BI** (built into Desktop) for better cartographic styling. Or use **Mapbox Visual** from AppSource for custom basemaps.

---

#### 9. Add the DAX Measures

Go to **Modeling** → **New Measure** and add each measure from the DAX block above:

1. `Cluster Count` — for dynamic KPI if you add slicers later
2. `Avg Silhouette` — for filtered average in tooltips
3. `PM25 Above Threshold` — creates an "Above"/"Below" column for conditional formatting
4. `Cluster Color` — use in **conditional formatting** rules for automatic coloring

---

#### 10. Final Polish

| Action | How |
|--------|-----|
| **Page titles** | Text box → "Cluster Overview & Profiles" (Page 1), "Method Validation & Spatial View" (Page 2) |
| **Theme colors** | **View** → **Themes** → **Customize** → Primary = `#6f070f`, Secondary = `#87512d`, Tertiary = `#003e2f` |
| **Fonts** | Headers: Segoe UI Semibold 14pt, Body: Segoe UI 10pt |
| **Slicer** | Add a **Slicer** with `cluster_name` on both pages for interactive filtering |
| **Background** | **Format** → **Canvas background** → color `#FFF8F1` (warm surface from our design system) |
| **Navigation** | **Insert** → **Buttons** → **Page navigator** for switching between pages |

> **💡 Teaching Moment:** The difference between a "report" and a "dashboard" in PowerBI: a **report** is what you build in Desktop (multiple pages, interactive). A **dashboard** is a single-page pinned view in the PowerBI Service (cloud). For the lab, we build a report with 2 pages.

---

## Lab Complete ✅

You've completed all 5 steps of the K-Means Clustering Lab:

| Step | Deliverable | Status |
|------|------------|--------|
| **Step 1** | Data Understanding & Clustering Opportunities | ✅ Explored 48K rows, identified 4 clustering features |
| **Step 2** | Contextual Visuals | ✅ Scatter plots, correlation heatmaps, radar profiles |
| **Step 3** | Didactic K-Means Implementation | ✅ Elbow, silhouette, PCA, cluster-informed quadrants |
| **Step 4** | React Dashboard Page | ✅ Built separately at `/analytics/clustering` |
| **Step 5** | PowerBI Data Export | ✅ 7 CSV files exported to `data/powerbi/` |

### Key Takeaway

Three clusters emerge from Chinatown's 12 open-space sites:
- 🔴 **8 sites** share a "High Pollution + Humid" profile — the dominant urban condition
- 🟡 **Castle Square** stands alone as a micro-heat island
- 🟢 **3 sites** form a "Cleaner & Drier" refuge cluster

This 8-1-3 distribution is the most actionable finding: most of Chinatown's open spaces face similar environmental challenges, with only a few exceptions that can inform better park design.

---
*HEROS K-Means Clustering Lab · Tufts University · Environmental Data Science*